In [ ]:
# 데이터 생성 : csv파일을 불러옴
import pandas as pd

df = pd.read_csv('../data/csv/sentiment_data_long.csv')

In [ ]:
# 독립, 종속변수 분리
X = df['sentence']
y = df['label']

In [ ]:
# 훈련, 테스트 분리 
DATA_SIZE = 1000
TRAIN_SIZE = int(DATA_SIZE * 0.8)

X_train, X_test = X[:TRAIN_SIZE], X[TRAIN_SIZE: ]
y_train, y_test = y[:TRAIN_SIZE], y[TRAIN_SIZE: ]

In [ ]:
# 형태소 분리 시 모든 형태소를 포함
from konlpy.tag import Okt
cleaned_sentence = []
def get_preprocessing(sentence):
	okt = Okt()
	result = okt.pos(sentence, stem=True) # 문장을 형태소별로 나눔. 단, 원형으로
	words = [word for word, pos in result]
	return " ".join(words)
# 분리한 단어들을 합침
# print(X_train[0])
# get_preprocessing(X_train[0])
X_train = X_train.apply(get_preprocessing)
X_test = X_test.apply(get_preprocessing)


In [ ]:

# 벡터화 
from tensorflow.keras import layers, models
vectorize_layer = layers.TextVectorization(
	max_tokens=1000,
	output_mode="int",
	output_sequence_length=10
)
vectorize_layer.adapt(X_train.tolist())

In [ ]:

import tensorflow as tf

# 모델 설계
model = models.Sequential([
	layers.Input((1, ), dtype=tf.string),
	vectorize_layer, 
	layers.Embedding(input_dim=1000, output_dim=64),
	layers.LSTM(32),
	layers.Dense(1, activation='sigmoid')#출력층
])

TypeError: Embedding.__init__() missing 2 required positional arguments: 'input_dim' and 'output_dim'

In [ ]:

model.compile(
	optimizer='adam', 
	loss='binary_crossentropy', 
	metrics=['accuracy']
) 

NameError: name 'model' is not defined

In [ ]:

# 평가
_, acc = model.evaluate(X_test.values, y_test)
print(f'정확도 : {acc}')

In [ ]:
# 예측
# "너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요"
# 오늘 날이 좋고 기분이 좋아요
# 듣고 있는 음악이 슬퍼서 우울해요
import numpy as np
sentences = [
	"너무 피곤하고 듣고 있는 음악은 별로지만 기분이 좋아요",
	"지루한 장면이 반복되어서 스토리가 너무 개연성이 없고 결말이 허무하기 짝이 없다"
]
# 학습 데이터가 긍정=> 부정, 부정=>긍정인 데이터들이어서 
# 긍정 문구가 있으면 부정으로 될거라고 예측
# 부정문구가 있으면 긍정으로 될거라고 예측

# 머신러닝/단순한 MLP 방식의 문장 분석
# 	=> 각 단어

predictions = model.predict(np.array(sentences,dtype=object))
predictions
